# T2 Data access and profiling

In [ ]:
import os
os.environ["AWS_PROFILE"] = "javier-whizdom-prod-ds"

import boto3
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BUCKET = "whizdomai-eu-central-1-650177547431-datalake-gmntc-prod"


def s3_duckdb():
    """DuckDB connection with the credentials of the SSO profile already loaded."""
    creds = boto3.Session(profile_name="javier-whizdom-prod-ds").get_credentials().get_frozen_credentials()
    con = duckdb.connect()
    con.sql("INSTALL httpfs; LOAD httpfs;")

    con.sql("SET memory_limit='4GB'")
    con.sql("SET threads=2")
    con.sql(f"""
        CREATE OR REPLACE SECRET s3_secret (
            TYPE S3, KEY_ID '{creds.access_key}', SECRET '{creds.secret_key}',
            SESSION_TOKEN '{creds.token}', REGION 'eu-central-1'
        );
    """)
    return con

## Read access: operator tables in the `10-landing` layer (kafka-sink)

We checked access on 2026-09-10, to `s3://whizdomai-eu-central-1-650177547431-datalake-gmntc-prod/org/10-landing/kafka-sink/`.

There are two operators. Each one has the same five tables: `primus/` and `secundus/`.

| Business table | Real S3 prefix | Why we chose this mapping |
|---|---|---|
| `player` | `whizdomai-players` | Profile and registration columns: `firstname`, `birthdate`, `regdate`, `kycStatus`. |
| `bet` | `whizdomai-transactions` | The prefix name sounds like money transactions, but it has `gameId`, `gameTranId`, `tranType`, `amountReal` and `balanceReal`. This is the game and betting ledger. |
| `transaction` | `whizdomai-payments` | The prefix name sounds like generic payments, but it has `requestedAmount`, `fee`, `accountNumber` and `transactionMethod`. This is deposits and withdrawals. |
| `bonus` | `whizdomai-bonuses` | Columns about bonus exposure per player: `triggerDate`, `wagerRequirement`, `releasedBonus`. |
| `session` | Does not exist | There is no session or login prefix under `10-landing/kafka-sink`, for `primus` or `secundus`. We also checked the gold layer, and it is not there either. |


In [ ]:
LANDING_LEVEL = "org/10-landing/kafka-sink"
OPERATORS = ["primus", "secundus"]


LANDING_TABLES = {
    "player": "whizdomai-players",
    "bet": "whizdomai-transactions",
    "transaction": "whizdomai-payments",
    "bonus": "whizdomai-bonuses",
}

SNAPSHOT_DAY = ("2026", "09", "10")


def run_landing_query(query, tables, operator, year="*", month="*", day="*"):
    """Run a query on S3 landing Parquet files. Do not download any file to disk.

    Args:
        query (str): The SQL query to run.
        tables (list[str]): List of landing table prefixes, e.g. ["whizdomai-players"].
        operator (str): "primus" or "secundus".
        year (str, optional): Year partition. Use "*" for all years. Default is "*".
        month (str, optional): Month partition. Use "*" for all months. Default is "*".
        day (str, optional): Day partition. Use "*" for all days. Default is "*".

    Returns:
        pd.DataFrame: The query result.

    Note:
        Landing is the raw Kafka-Connect S3 sink. The Gold layer stores each day
        in one file, `data.parquet`. Landing is different: it has many small
        `*.snappy.parquet` files per day, one for each hour. So we need a glob
        pattern (`*`) to read them all.
    """
    con = s3_duckdb()
    for t in tables:
        path = f"s3://{BUCKET}/{LANDING_LEVEL}/{operator}/{t}/{year}/{month}/{day}/*.snappy.parquet"
        view = t.replace("-", "_")
        con.sql(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM read_parquet('{path}', union_by_name=True)")
    result = con.sql(query).df()
    con.close()
    return result

Access check: one `COUNT` per table and operator, for the snapshot day. We do not load any rows into memory.

In [ ]:
access_check = []
for operator in OPERATORS:
    for business_name, landing_table in LANDING_TABLES.items():
        view = landing_table.replace("-", "_")
        try:
            n_rows = run_landing_query(
                f"SELECT count(*) AS n FROM {view}",
                tables=[landing_table], operator=operator,
                year=SNAPSHOT_DAY[0], month=SNAPSHOT_DAY[1], day=SNAPSHOT_DAY[2],
            )["n"].iloc[0]
            access_check.append((operator, business_name, landing_table, "OK", n_rows))
        except Exception as e:
            access_check.append((operator, business_name, landing_table, f"FAIL: {e}", None))

access_check = pd.DataFrame(access_check, columns=["operator", "business_table", "landing_table", "status", "rows"])
access_check

,operator,business_table,landing_table,status,rows
0,primus,player,whizdomai-players,OK,13493
1,primus,bet,whizdomai-transactions,OK,13872583
2,primus,transaction,whizdomai-payments,OK,84570
3,primus,bonus,whizdomai-bonuses,OK,60353
4,secundus,player,whizdomai-players,OK,11326
5,secundus,bet,whizdomai-transactions,OK,8508825
6,secundus,transaction,whizdomai-payments,OK,49330
7,secundus,bonus,whizdomai-bonuses,OK,48660


**Result.** We confirmed read access for all 8 table × operator combinations (4 tables, 2 operators). `session` is not in this list because it does not exist as a table in this layer (see the section above).

## Data quality: `10-landing` layer

Every calculation runs in DuckDB, over 100% of the rows in the chosen period. `COUNT`, `glob` and `SUMMARIZE` scan all the data in streaming mode, and only send the summary back to pandas.

We split the profiling into two time periods. The cost of each check decides the period:

- **Date range and gaps** (`HISTORY_START`–`HISTORY_END`): covers all available history, from `2026-06-18` to `2026-09-29`. This check is cheap: `glob()` only lists the files for each day, it does not read them (about 0.3–0.7 seconds per day, no matter the table size). This lets us cover about four months, four tables and two operators in a few minutes.
- **Nulls, row count and grain signal** (`QUALITY_START`–`QUALITY_END`): uses a default 7-day window (`2026-09-23` to `2026-09-29`). This check is more expensive, because `SUMMARIZE` reads every column. For `bet` (`whizdomai-transactions`) it took about 162 seconds per operator, per day scanned. We could widen this window to the full history, but for `bet` alone that would take several hours.


In [ ]:
import datetime as dt

GRAIN_CANDIDATE_KEYS = {
    "player": ["uuid", "partyId"],
    "bet": ["transactionId", "partyId"],
    "transaction": ["paymentId", "partyId"],
    "bonus": ["changeId", "id", "partyId"],
}


HISTORY_START = dt.date(2026, 6, 18)
HISTORY_END = dt.date(2026, 9, 29)
HISTORY_DAYS = [HISTORY_START + dt.timedelta(days=i) for i in range((HISTORY_END - HISTORY_START).days + 1)]


QUALITY_START = dt.date(2026, 9, 23)
QUALITY_END = dt.date(2026, 9, 29)
QUALITY_DAYS = [QUALITY_START + dt.timedelta(days=i) for i in range((QUALITY_END - QUALITY_START).days + 1)]


def landing_paths(operator, landing_table, days):
    return [
        f"s3://{BUCKET}/{LANDING_LEVEL}/{operator}/{landing_table}/{d.year}/{d.month:02d}/{d.day:02d}/*.snappy.parquet"
        for d in days
    ]


def daily_presence(business_name, operator, days):
    """Check, for each day, if any file exists in S3. Do not read the file content.

    This is a light check. It only lists file names with `glob()`. It does not
    open or read the files. We use it to find the real date range, and to
    find gap days (days with no data).

    Args:
        business_name (str): Business table name, e.g. "player".
        operator (str): "primus" or "secundus".
        days (list[date]): List of days to check.

    Returns:
        list[dict]: One row per day, with the operator, table name, date, and file count.

    Note:
        Each check takes about 0.3-0.7 seconds per day. Table size does not
        matter, because we only list files, we never read them. This makes it
        cheap enough to run over the full history. A day with 0 files can
        mean two things: a real gap, or a day outside the table's real
        range. The code that calls this function decides which one it is.
    """
    landing_table = LANDING_TABLES[business_name]
    con = s3_duckdb()
    rows = []
    for d in days:
        path = f"s3://{BUCKET}/{LANDING_LEVEL}/{operator}/{landing_table}/{d.year}/{d.month:02d}/{d.day:02d}/*.snappy.parquet"
        n_files = con.sql(f"SELECT count(*) FROM glob('{path}')").fetchone()[0]
        rows.append({"operator": operator, "business_table": business_name, "date": d, "n_files": n_files})
    con.close()
    return rows


def column_profile(business_name, operator, days):
    """Run `SUMMARIZE` over every row in `days`. One pass, no sampling.

    This gives the exact row count and the exact null % for each column. It
    also gives an approximate count of unique values per column (using
    HyperLogLog). We use that approximate count as a signal for the table's grain.
    """
    landing_table = LANDING_TABLES[business_name]
    view = landing_table.replace("-", "_")
    paths = landing_paths(operator, landing_table, days)
    con = s3_duckdb()
    con.sql(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM read_parquet({paths}, union_by_name=True)")
    summary = con.sql(f"SUMMARIZE SELECT * FROM {view}").df()
    con.close()
    summary.insert(0, "operator", operator)
    summary.insert(1, "business_table", business_name)
    return summary

### Primary key uniqueness (grain)

In [12]:
exact_grain_rows = []
for operator in OPERATORS:
    for business_name, keys in GRAIN_CANDIDATE_KEYS.items():
        landing_table = LANDING_TABLES[business_name]
        view = landing_table.replace("-", "_")
        distinct_exprs = ", ".join(f'count(DISTINCT "{k}") AS n_{k}' for k in keys)
        result = run_landing_query(
            f"SELECT count(*) AS n_rows, {distinct_exprs} FROM {view}",
            tables=[landing_table], operator=operator,
            year=SNAPSHOT_DAY[0], month=SNAPSHOT_DAY[1], day=SNAPSHOT_DAY[2],
        )
        for k in keys:
            exact_grain_rows.append({
                "operator": operator,
                "business_table": business_name,
                "column_name": k,
                "n_distinct": int(result[f"n_{k}"].iloc[0]),
                "n_rows": int(result["n_rows"].iloc[0]),
            })

exact_grain_check = pd.DataFrame(exact_grain_rows)
exact_grain_check["uniqueness_ratio"] = (exact_grain_check["n_distinct"] / exact_grain_check["n_rows"]).round(4)
exact_grain_check = exact_grain_check.sort_values(
    ["business_table", "operator", "uniqueness_ratio"], ascending=[True, True, False]
)
print(f"Exact grain check for {SNAPSHOT_DAY[0]}-{SNAPSHOT_DAY[1]}-{SNAPSHOT_DAY[2]}")
exact_grain_check

Exact grain check for 2026-09-10


,operator,business_table,column_name,n_distinct,n_rows,uniqueness_ratio
2,primus,bet,transactionId,13872583,13872583,1.0000
3,primus,bet,partyId,16862,13872583,0.0012
11,secundus,bet,transactionId,8508825,8508825,1.0000
12,secundus,bet,partyId,11401,8508825,0.0013
6,primus,bonus,changeId,60353,60353,1.0000
7,primus,bonus,id,28713,60353,0.4758
8,primus,bonus,partyId,12548,60353,0.2079
15,secundus,bonus,changeId,48660,48660,1.0000
16,secundus,bonus,id,22972,48660,0.4721
17,secundus,bonus,partyId,9755,48660,0.2005


| Business table | Grain | Key that proves it |
|---|---|---|
| `player` | One row per player change event. The key is `partyId`, and it repeats. `uuid` does not identify the player — it identifies the event itself, and it is 100% unique. | `uuid` unique, `partyId` repeats |
| `bet` | One row per bet or transaction. The key is `transactionId`, 100% unique. | `transactionId` unique |
| `transaction` (`whizdomai-payments`) | One row per payment status change, not one row per payment. `paymentId` repeats. | `paymentId` repeats |
| `bonus` | One row per bonus status-change event. The key is `changeId`, 100% unique. `id` identifies the bonus itself, and it repeats. | `changeId` unique, `id` repeats |

**Result.** Each table has one key that is 100% unique, and this is the same in both operators: `player.uuid`, `bet.transactionId`, `bonus.changeId`. No candidate key in `transaction` (`whizdomai-payments`) is unique: `paymentId` only identifies about 58% of the rows in both operators (ratio 0.58–0.59). This confirms that the table stores payment status-change events, not one row per payment.

### Date range and gaps

In [ ]:
presence_rows = []
for operator in OPERATORS:
    for business_name in LANDING_TABLES:
        presence_rows += daily_presence(business_name, operator, HISTORY_DAYS)

presence = pd.DataFrame(presence_rows)
presence["has_data"] = presence["n_files"] > 0

summary_rows = []
for (operator, business_name), grp in presence.groupby(["operator", "business_table"]):
    with_data = grp[grp["has_data"]]
    if with_data.empty:
        summary_rows.append({"operator": operator, "business_table": business_name,
                              "date_min": None, "date_max": None, "n_days_with_data": 0,
                              "n_gap_days": None, "gap_dates": []})
        continue
    date_min, date_max = with_data["date"].min(), with_data["date"].max()

    within_range = grp[(grp["date"] >= date_min) & (grp["date"] <= date_max)]
    gaps = sorted(within_range.loc[~within_range["has_data"], "date"].tolist())
    summary_rows.append({
        "operator": operator, "business_table": business_name,
        "date_min": date_min, "date_max": date_max,
        "n_days_with_data": int(with_data.shape[0]),
        "n_gap_days": len(gaps), "gap_dates": gaps,
    })

date_range_gaps = pd.DataFrame(summary_rows)
date_range_gaps

,operator,business_table,date_min,date_max,n_days_with_data,n_gap_days,gap_dates
0,primus,bet,2026-06-18,2026-09-29,104,0,[]
1,primus,bonus,2026-06-18,2026-09-29,104,0,[]
2,primus,player,2026-06-18,2026-09-29,104,0,[]
3,primus,transaction,2026-06-18,2026-09-29,104,0,[]
4,secundus,bet,2026-06-18,2026-09-29,104,0,[]
5,secundus,bonus,2026-06-18,2026-09-29,104,0,[]
6,secundus,player,2026-06-18,2026-09-29,104,0,[]
7,secundus,transaction,2026-06-18,2026-09-29,104,0,[]


**Result.** 0 gaps in all 8 table × operator combinations, over 104 days of available history (`2026-06-18` to `2026-09-29`). All 4 tables start and end on the same day, in both operators.

### Nulls, row count and approximate grain

In [ ]:
from IPython.display import display


column_profiles = {}
for operator in OPERATORS:
    for business_name in LANDING_TABLES:
        column_profiles[(operator, business_name)] = column_profile(business_name, operator, QUALITY_DAYS)

all_profiles = pd.concat(column_profiles.values(), ignore_index=True)

row_counts = (
    all_profiles.groupby(["operator", "business_table"])["count"]
    .first()
    .rename("n_rows_in_window")
    .reset_index()
)

null_summary = all_profiles[all_profiles["null_percentage"] > 0][
    ["operator", "business_table", "column_name", "null_percentage"]
].sort_values("null_percentage", ascending=False)

candidate_cols = {k for keys in GRAIN_CANDIDATE_KEYS.values() for k in keys}
grain_check = all_profiles[all_profiles["column_name"].isin(candidate_cols)][
    ["operator", "business_table", "column_name", "approx_unique", "count"]
].copy()
grain_check["uniqueness_ratio"] = (grain_check["approx_unique"] / grain_check["count"]).round(3)
grain_check = grain_check.sort_values(["business_table", "operator", "uniqueness_ratio"], ascending=[True, True, False])

print(f"Quality window: {QUALITY_START} to {QUALITY_END}")
display(row_counts)
display(null_summary)
display(grain_check)

Quality window: 2026-09-23 to 2026-09-29


,operator,business_table,n_rows_in_window
0,primus,bet,88708725
1,primus,bonus,374891
2,primus,player,88630
3,primus,transaction,450996
4,secundus,bet,48981846
5,secundus,bonus,262739
6,secundus,player,86230
7,secundus,transaction,276373


,operator,business_table,column_name,null_percentage
23,primus,player,birthCity,100.00
22,primus,player,birthCountry,100.00
122,secundus,player,unit,100.00
29,primus,player,iban,100.00
137,secundus,player,lastKycRequestedDate,99.99
...,...,...,...,...
172,secundus,bet,gameTranId,0.06
67,primus,bet,platformTranId,0.05
68,primus,bet,gameTranId,0.02
69,primus,bet,gameId,0.02


,operator,business_table,column_name,approx_unique,count,uniqueness_ratio
41,primus,bet,transactionId,115980111,88708725,1.307
40,primus,bet,uuid,82377306,88708725,0.929
44,primus,bet,partyId,47551,88708725,0.001
144,secundus,bet,uuid,53423817,48981846,1.091
145,secundus,bet,transactionId,41867081,48981846,0.855
148,secundus,bet,partyId,24223,48981846,0.000
86,primus,bonus,changeId,359583,374891,0.959
88,primus,bonus,id,153488,374891,0.409
89,primus,bonus,partyId,47740,374891,0.127
190,secundus,bonus,changeId,251842,262739,0.959


**Result.** 7-day window (`2026-09-23` to `2026-09-29`), both operators: `bet` has almost all the volume (88.7M rows in `primus`, 49.0M in `secundus`), compared to tens or hundreds of thousands in the other tables. Most important nulls: in `player`, address and KYC columns that few players fill in (`birthCity`, `birthCountry`, `iban`, `nationality`) have 95–100% nulls. This makes sense if most players have not finished verification. In `bet`, almost every column is full, except `rollbackTranId` (this only applies to reversed bets). The `uniqueness_ratio` in this table comes from `approx_unique`, a column from `SUMMARIZE`, and it is only an estimate: for `bet`/`primus`/`transactionId` it shows 1.307, which is impossible for a real count. This is why we also use the exact grain check above, which is reliable.

### Schema

This reuses the `SUMMARIZE` result from the cell above (same 7-day window): column name, type and null % for each table. Before we list it, we compare the set of columns and types between `primus` and `secundus`, to check there is no structural difference between operators.

In [ ]:
from IPython.display import display

mismatched = [
    business_name for business_name in LANDING_TABLES
    if set(zip(
        all_profiles.loc[(all_profiles.business_table == business_name) & (all_profiles.operator == OPERATORS[0]), "column_name"],
        all_profiles.loc[(all_profiles.business_table == business_name) & (all_profiles.operator == OPERATORS[0]), "column_type"],
    )) != set(zip(
        all_profiles.loc[(all_profiles.business_table == business_name) & (all_profiles.operator == OPERATORS[1]), "column_name"],
        all_profiles.loc[(all_profiles.business_table == business_name) & (all_profiles.operator == OPERATORS[1]), "column_type"],
    ))
]
print("Schema differs between operators in:", mismatched or "no tables")

for business_name in LANDING_TABLES:
    print(f"\n--- {business_name} ---")
    display(
        all_profiles[(all_profiles.business_table == business_name) & (all_profiles.operator == OPERATORS[0])]
        [["column_name", "column_type", "null_percentage"]]
        .sort_values("column_name")
        .reset_index(drop=True)
    )

## Summary: shape of the data

Built from the DataFrames we already computed above (`exact_grain_check`, `date_range_gaps`, `row_counts`, `null_summary`). This does not touch S3 again.

In [ ]:
summary_rows = []
for business_name in LANDING_TABLES:
    pk_per_op = (
        exact_grain_check[exact_grain_check.business_table == business_name]
        .groupby("column_name")["uniqueness_ratio"].min()
    )
    pk_cols = pk_per_op[pk_per_op == 1.0].index.tolist()

    dr = date_range_gaps[date_range_gaps.business_table == business_name]
    date_min, date_max = dr["date_min"].min(), dr["date_max"].max()
    n_gaps = int(dr["n_gap_days"].sum())

    total_rows_window = int(row_counts[row_counts.business_table == business_name]["n_rows_in_window"].sum())

    top_nulls = (
        null_summary[null_summary.business_table == business_name]
        .groupby("column_name")["null_percentage"].max()
        .sort_values(ascending=False)
        .head(3)
    )
    top_nulls_str = ", ".join(f"{name} ({pct:.1f}%)" for name, pct in top_nulls.items()) or "no relevant nulls"

    summary_rows.append({
        "table": business_name,
        "primary key": ", ".join(pk_cols) if pk_cols else "no candidate key is 100% unique",
        "date range": f"{date_min} to {date_max}",
        "gap days": n_gaps,
        "rows (7d window, 2 operators)": total_rows_window,
        "columns with most nulls": top_nulls_str,
    })

data_shape_summary = pd.DataFrame(summary_rows)
data_shape_summary

**Read.** All 4 tables cover the same range (`2026-06-18` to `2026-09-29`, 0 gaps) in both operators. Three tables have one key that is 100% unique per row (`player.uuid`, `bet.transactionId`, `bonus.changeId`); `transaction` does not, because its grain is a payment status change, not a payment. `bet` holds almost all the volume (~137.7M rows in just the 7-day window, against hundreds of thousands in the rest), and it is the only table that forces us to use SQL pushdown instead of loading the full DataFrame. The most notable finding is 98.3% nulls in `transaction.requestedAmount`, worth confirming with the data team. `session` has no table in this layer.